
1. 가상환경 준비 
```
conda create -n rkllm python=3.11 -y
```

2. git clone 
```
git clone https://github.com/airockchip/rknn-llm.git
```

3. 설치 파일 확인 
```
cd ~/rknn-llm/rkllm-toolkit/packages

ls
```

4. 자기 파이썬 환경에 맞게 설치 진행 

```
pip3 install rkllm_toolkit-x.x.x-cp3xx-cp3xx-linux_x86_64.whl
```
예:

파이썬 3.11 이라면 

```
pip3 install rkllm_toolkit-1.2.2-cp311-cp311-linux_x86_64.whl
```


In [19]:
from rkllm.api import RKLLM

rkllm = RKLLM()

INFO: rkllm-toolkit version: 1.2.2


# 1. gguf 파일에서 rkllm 파일로 변환

In [20]:
 
upload_repo_id = 'AeiROBOT/EDIE_qwen2.5-0.5B-Instruct_EDIE_MIND_Emotion' 

model_path = upload_repo_id + '-gguf'

load_model = f"{model_path}/EDIE_qwen2.5-0.5B-Instruct_EDIE_MIND_Emotion-merged.F16.gguf"

save_path = upload_repo_id + "-rkllm" 

export_model_name = 'Qwen2.5-0.5b-instruct_w8a8_g128_gguf.rkllm'
export_model = f'{save_path}/{export_model_name}'

import os
os.makedirs(f"./{save_path}", exist_ok=True)

In [21]:
ret = rkllm.load_gguf(model = load_model)

if ret != 0:
    print('모델 로드 실패!')

In [22]:
ret = rkllm.build(
    do_quantization=False,            # 양자화 수행 여부
    optimization_level=1,            # 정밀도 최적화
    quantized_dtype='w8a8',          # 양자화 타입
    quantized_algorithm="normal",    # 알고리즘 : normal, grq, gdq
    num_npu_core=3,                  # NPU 코어 개수 (RK3588)
    extra_qparams=None,              # qparams 캐시 없음
    dataset="quant_data.json",       # 양자화용 데이터셋
    hybrid_rate=0.5,                   # 하이브리드 양자화 비율
    target_platform='rk3588'         # 대상 플랫폼
)

if ret != 0:
    print('모델 빌드 실패!')

In [23]:
ret = rkllm.export_rkllm(export_path = export_model)
if ret != 0:
    print('모델 내보내기 실패!')

Converting model: 100%|██████████| 290/290 [00:05<00:00, 55.68it/s]
INFO: Setting max_context_limit to 4096


INFO: Exporting the model, please wait ....
[=================================================>] 513/513 (100%)


INFO: Model has been saved to AeiROBOT/EDIE_qwen2.5-0.5B-Instruct_EDIE_MIND_Emotion-rkllm/Qwen2.5-0.5b-instruct_w8a8_g128_gguf.rkllm!



`/home/khw/Workspace/LLM/AeiROBOT/EDIE_qwen2.5-0.5B-Instruct_EDIE_MIND_Emotion-gguf-rkllm` 폴더가 생성되었고 

그 안에 'Qwen2.5-0.5b-instruct_w8a8_g128_gguf.rkllm' 이름의 파일이 생성되었음을 확인할 수 있습니다.


# 2. 파인튜닝 모델 직접 변환

In [24]:
from rkllm.api import RKLLM

rkllm = RKLLM()

INFO: rkllm-toolkit version: 1.2.2


In [25]:
# HuggingFace 형식으로 로드 (파인튜닝된 모델 폴더 경로)

upload_repo_id = 'AeiROBOT/EDIE_qwen2.5-0.5B-Instruct_EDIE_MIND_Emotion' 

model_path = upload_repo_id + '-merged'

save_path = upload_repo_id + "-rkllm" 

export_model_name = 'Qwen2.5-0.5b-instruct_w8a8_g128_direct_qaunt.rkllm'
export_model = f'{save_path}/{export_model_name}'

import os
os.makedirs(f"./{save_path}", exist_ok=True)

In [26]:

ret = rkllm.load_huggingface(model=model_path, device='cuda')  # 또는 'cpu'

if ret != 0:
    print('모델 로드 실패!')
    exit()

ret = rkllm.build(
    do_quantization=True,            
    optimization_level=1,            
    quantized_dtype='w8a8_g128',         # w8a8 : / w8a8_g128 : 이게 정확도가 높지만, 연산 부하도 높음
    quantized_algorithm="normal",    
    num_npu_core=3,                  
    hybrid_rate=0.5,                
    dataset="quant_data.json",       # 양자화용 데이터셋 
    target_platform='rk3588'         
)


# # 디버깅을 위해 양자화를 끄고 테스트 (정확도가 돌아오는지 확인)
# ret = rkllm.build(
#     do_quantization=False,  # 우선 False로 해서 변환 자체의 문제인지 확인!
#     optimization_level=0,
#     target_platform='rk3588'
# )

INFO: Mixed types include w8a8 and W8A8_G128, with W8A8_G128 being the primary type.
Calculating quantization sensitivity: 100%|██████████| 24/24 [02:24<00:00,  6.01s/it]
INFO: block_id:0, sensitivity:[0.5060164326569065, 0.2643648264929652], delta:0.24165160616394132
INFO: block_id:1, sensitivity:[0.21951159369200468, 0.07366177832591347], delta:0.1458498153660912
INFO: block_id:2, sensitivity:[23.21028856933117, 0.4326269645243883], delta:22.77766160480678
INFO: block_id:3, sensitivity:[14.266181506216526, 2.7548056934028864], delta:11.51137581281364
INFO: block_id:4, sensitivity:[0.22522698959801346, 0.11351592309074476], delta:0.1117110665072687
INFO: block_id:5, sensitivity:[1.051424888893962, 0.13455076684476808], delta:0.9168741220491938
INFO: block_id:6, sensitivity:[0.1661920758197084, 0.04649078025249764], delta:0.11970129556721076
INFO: block_id:7, sensitivity:[0.1586001893156208, 0.05014457821380347], delta:0.10845561110181734
INFO: block_id:8, sensitivity:[0.30703363270731

In [27]:
ret = rkllm.export_rkllm(export_path = export_model)
if ret != 0:
    print('모델 내보내기 실패!')

INFO: Setting token_id of eos to 151645
INFO: Setting add_bos_token to False
Converting model: 100%|██████████| 291/291 [00:00<00:00, 1514320.67it/s]
INFO: Setting max_context_limit to 4096


INFO: Exporting the model, please wait ....
[=================================================>] 513/513 (100%)========>             ] 374/513 (72%)


INFO: Model has been saved to AeiROBOT/EDIE_qwen2.5-0.5B-Instruct_EDIE_MIND_Emotion-rkllm/Qwen2.5-0.5b-instruct_w8a8_g128_direct_qaunt.rkllm!


# Upload model to HuggingFace

In [28]:
from huggingface_hub import login, create_repo, upload_folder
import os
from dotenv import load_dotenv
load_dotenv()

True

In [29]:
HF_TOKEN=os.getenv("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

In [30]:
create_repo(
    repo_id=save_path,
    token=HF_TOKEN,
    private=True,
    exist_ok=True,
)

# 3) 폴더 업로드
upload_folder(
    repo_id=save_path,
    folder_path=save_path,
    token=HF_TOKEN,
)


print(f"[done] https://huggingface.co/{save_path}")

Processing Files (2 / 2): 100%|██████████| 2.13GB / 2.13GB, 4.34MB/s  
New Data Upload: 100%|██████████|  716MB /  716MB, 4.34MB/s  


[done] https://huggingface.co/AeiROBOT/EDIE_qwen2.5-0.5B-Instruct_EDIE_MIND_Emotion-rkllm


In [ ]:

# from rkllm.api import RKLLM

# rkllm = RKLLM()

# # HuggingFace 형식으로 로드 (파인튜닝된 모델 폴더 경로)
# model_path = "/home/khw/Arobot/edie8_llm_simple/AeiROBOT/EDIE_qwen2.5-1.5B-Instruct_Emotion-QA-merged"

# ret = rkllm.load_huggingface(model=model_path, device='cuda')  # 또는 'cpu'

# if ret != 0:
#     print('모델 로드 실패!')
#     exit()

# ret = rkllm.build(
#     do_quantization=True,            
#     optimization_level=1,            
#     quantized_dtype='w8a8_g128',         # w8a8 : / w8a8_g128 : 이게 정확도가 높지만, 연산 부하도 높음
#     quantized_algorithm="normal",    
#     num_npu_core=3,                  
#     hybrid_rate=0.5,                 
#     target_platform='rk3588'         
# )

/home/khw/miniconda3/envs/RKLLM/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/khw/miniconda3/envs/RKLLM/lib/python3.11/site-packages/rkllm/api/rkllm.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  self.base = RKLLMBase(v)
INFO: rkllm-toolkit version: 1.2.2
Loading checkpoint shards: 100%|██████████| 2/2 [00:16<00:00,  8.03s/it]
INFO: Mixed types include w8a8 and W8A8_G128, with W8A8_G128 being the primary type.
Calculating quantization sensitivity: 100%|██████████| 36/36 [01:37<00:00,  2.71s/it]
INFO: block_id:0, sensitivity:[0.022119161381851882, 0.002971875275761704], delta:0.019147286106090178


In [ ]:
# save_path = 'AeiROBOT/EDIE_qwen2.5-1.5B-Instruct_Emotion-QA-merged-rkllm'
# export_model_name = 'Qwen2.5-1.5b-instruct_w8a8_g128.rkllm'
# export_model = f'{save_path}/{export_model_name}'

# import os
# os.makedirs(f"./{save_path}", exist_ok=True)

In [ ]:
# ret = rkllm.export_rkllm(export_path = export_model)
# if ret != 0:
#     print('모델 내보내기 실패!')

INFO: Setting token_id of eos to 151645
INFO: Setting add_bos_token to False
Converting model: 100%|██████████| 435/435 [00:00<00:00, 1774827.08it/s]
INFO: Setting max_context_limit to 4096


INFO: Exporting the model, please wait ....
[=================================================>] 765/765 (100%)


INFO: Model has been saved to AeiROBOT/baseline_qwen2.5-3.5B-Instruct_Emotion-QA-rkllm2/Qwen2.5-3b-instruct_w8a8_g128.rkllm!


In [ ]:



# create_repo(
#     repo_id=save_path,
#     token=HF_TOKEN,
#     private=False,
#     exist_ok=True,
# )

# # 3) 폴더 업로드
# upload_folder(
#     repo_id=save_path,
#     folder_path=save_path,
#     token=HF_TOKEN,
# )


# print(f"[done] https://huggingface.co/{save_path}")